# March Mania · Workspace and raw-data preflight

**Milestone 0 — synchronize, preserve, inspect. No training.**

This companion notebook stays outside the source repository. It does not replace the six canonical project notebooks. Complete the `inventory`, `sync`, optional `restore-data`, and `environment` terminal stages in **START_HERE.md** first. Choose **Python (March Mania)** as the kernel.

Best submitted Brier **0.1222672**; research target **0.1097454**. Those historical values are context, not results from this notebook. The feature-engineering research phase remains open.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, time
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, Markdown, FileLink

KIT = Path.cwd().resolve()
if not (KIT / "workspace_sync.py").is_file():
    KIT = Path.home() / "march_workspace_sync"
REPO = Path(os.environ.get("MARCH_REPO", str(Path.home() / "march-machine-learning-mania-2026"))).expanduser().resolve()
REPORTS = Path(os.environ.get("MARCH_SYNC_REPORTS", str(KIT / "reports"))).expanduser().resolve()
EXPECTED = "84b8fb36644a6558beded6dad84f5645ea4405d3"
assert (KIT / "workspace_sync.py").is_file(), "Open this notebook from the extracted kit folder."
assert (REPO / ".git").exists(), "Stop: repository not found. Check the space and path; do not delete anything."
print("Repository:", REPO)
print("Readiness reports:", REPORTS)
print("Kernel:", sys.executable)
figures = []

## 1. Verify synchronization evidence

Source identity, raw-data presence, and preservation are separate checks. A clean Git checkout alone cannot prove the raw dataset exists. Local stashes and recovery bundles are private and must not be added to public GitHub.

In [ ]:
sync_path = REPORTS / "sync.json"
assert sync_path.is_file(), "Run the terminal sync command first and inspect its receipt."
sync = json.loads(sync_path.read_text())
assert sync.get("status") == "PASS", "Synchronization stopped. Inspect the saved receipt before continuing."
assert sync.get("head") == EXPECTED, "The checkout differs from the verified reference. Review the SHA and CI before proceeding."
assert sync.get("tracked_matches_origin_main") is True
assert sync.get("raw_sha256_unchanged") is True
assert sync.get("private_metadata_unchanged") is True
checks = {key: sync.get(key) for key in [
    "head", "branch", "tracked_matches_origin_main", "raw_data_present",
    "raw_files_hashed", "raw_sha256_unchanged", "private_metadata_unchanged",
    "github_writes", "training_started"]}
display(pd.DataFrame({"check": checks.keys(), "observed": [str(x) for x in checks.values()]}))
print("Synchronization preserved existing bytes; subsequent data restoration is checked separately below.")

## 2. Audit the actual CSVs, with bounded and resumable execution

This stage streams file-level schema/coverage checks and SHA-256 values. It never fits a model. It reuses a per-file audit only when the bytes match. The stage limit is **600 seconds**, with **15-second heartbeats**. These are stop/progress settings, not runtime predictions.

A current Kaggle download can include 2026 tournament results. Those rows are flagged, not used as predictors or training labels for a 2026 forecast. This is a table-level preflight; per-team completeness and leakage tests remain mandatory when implementing features.

In [ ]:
command = [sys.executable, str(KIT / "workspace_sync.py"), "audit",
           "--repo", str(REPO), "--out", str(REPORTS),
           "--expected-commit", EXPECTED, "--max-seconds", "600"]
with subprocess.Popen(command, cwd=KIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
    try:
        for line in process.stdout:
            print(line.rstrip(), flush=True)
        return_code = process.wait(timeout=15)
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
if return_code not in (0, 2):
    raise RuntimeError("The data audit stopped. Read its STOP reason. Do not launch training.")
audit = json.loads((REPORTS / "data_audit.json").read_text())
summary = json.loads((REPORTS / "milestone_summary.json").read_text())
print("Ready for the next feature milestone:", audit["ready_for_next_feature_milestone"])
print("New experiments run:", summary["new_experiments_run"])
print("Optional files missing:", audit["optional_files_missing"])
print("Scope:", audit["availability_scope"])

## 3. Which seasons and sources are actually present?

Hover to inspect observed row counts. Blank entries mean no rows were found for that table/season; they must not be interpreted as zero-valued team features. The discontinued 2020 tournament is not treated as a missing modeling season.

In [ ]:
coverage = pd.read_csv(REPORTS / "data_coverage.csv")
assert not coverage.empty, "No season-level rows were audited. Stop and inspect raw inputs."
coverage["season"] = coverage["season"].astype(int)
wide = coverage.pivot(index="file", columns="season", values="rows").sort_index(axis=1)
fig = px.imshow(wide, aspect="auto", labels={"x": "Season", "y": "Official table", "color": "Observed rows"},
                title="Actual raw-data coverage · not feature validation")
fig.update_layout(height=max(460, 28 * len(wide) + 140), margin=dict(l=30, r=30, t=70, b=30))
fig.show()
figures.append(fig)
display(coverage.tail(12))

## 4. Ranking publication cutoffs

The source file can contain rankings published after the forecast deadline. Only observations at `RankingDayNum ≤ 132` are counted below. This chart counts distinct systems with any legal row; it does **not** establish edition consistency, freshness or per-team coverage. Those are further feature-stage checks.

In [ ]:
massey = coverage.loc[coverage["file"].eq("MMasseyOrdinals.csv")].sort_values("season")
if len(massey):
    fig = px.bar(massey, x="season", y="ranking_systems_pre132",
                 title="Men’s ranking systems with at least one pre-cutoff observation",
                 labels={"season": "Season", "ranking_systems_pre132": "Systems at or before day 132"})
    fig.show()
    figures.append(fig)
else:
    display(Markdown("**Massey data is absent. Do not silently claim rankings are available, and do not invent women’s rankings.**"))

## 5. Data volume, schema issues and outcome availability

This milestone hashes and inventories files but does not load serialized estimators. Old model/checkpoint directories stay in place. Missing private checkpoints are a recovery question, not a reason to retrain automatically.

In [ ]:
tables = pd.DataFrame(audit["tables"])
tables["size_mib"] = tables["bytes"] / 2**20
fig = px.bar(tables.sort_values("size_mib", ascending=False), x="file", y="size_mib",
             title="Audited source sizes", labels={"file": "Official CSV", "size_mib": "MiB"})
fig.update_layout(height=540, xaxis_tickangle=-55)
fig.show()
figures.append(fig)
display(tables[["file", "rows", "size_mib", "error", "has_2026_tournament_labels"]])
if tables["has_2026_tournament_labels"].any():
    display(Markdown("**2026 tournament labels are present. They are already-consumed retrospective diagnostics, not legal inputs for a 2026 pre-tournament predictor.**"))
print("Core files missing:", audit["core_files_missing"])
print("Schema issues:", audit["schema_errors"])
print("Modeled-season gaps:", audit["modeled_season_coverage_gaps"])

## 6. Resume feature research from evidence, not a larger column count

The existing bank already covers adjusted efficiency, Four Factors, Elo, multiple windows, opponent/venue context, coaches, conferences and rankings. Start with the actual fitted-column and family-ablation evidence. The table below is a proposed investigation queue, not a list of proven missing or useful features. Read **FEATURE_RESEARCH_PLAN.md** before designing the first bounded experiment.

In [ ]:
queue = pd.read_csv(KIT / "feature_investigation_queue.csv", dtype=str)
display(queue)
public_evidence = []
for pattern in ("**/feature_usage.csv", "**/screening_summary.csv", "**/selection_stability.csv", "**/feature_registry.csv"):
    public_evidence.extend((REPO / "reports").glob(pattern))
print("Matching committed/local public evidence files:")
for path in sorted(set(public_evidence)):
    print(path.relative_to(REPO))
if not public_evidence:
    print("No matching CSVs found under reports. Consult committed feature-store docs and version-pinned artifact manifests before any refit.")

## 7. Save the milestone, then stop before fitting

The HTML report embeds Plotly so it can be viewed without an external plotting service. The JSON summary contains only readiness/file-level evidence and the historical score context. Do not upload the private recovery directory or credentials.

A green gate below means **setup is ready for the next investigation**. It is not proof of feature improvement, full historical checkpoint restoration or a better leaderboard score.

In [ ]:
import html
REPORTS.mkdir(parents=True, exist_ok=True)
parts = ["<!doctype html><html><head><meta charset='utf-8'><title>March Mania · Workspace preflight</title></head><body>",
         "<h1>March Mania · Workspace and raw-data preflight</h1>",
         "<p>No model training. Table-level readiness and feature-research queue.</p>",
         "<pre>" + html.escape(json.dumps({k: summary[k] for k in ["head", "ready_for_next_feature_milestone", "core_files_missing", "schema_errors", "new_experiments_run"]}, indent=2)) + "</pre>"]
for i, fig in enumerate(figures):
    parts.append(pio.to_html(fig, full_html=False, include_plotlyjs=True if i == 0 else False))
parts += ["<h2>Next investigations · not validated feature gains</h2>", queue.to_html(index=False, escape=True), "</body></html>"]
(REPORTS / "workspace_audit.html").write_text("\n".join(parts), encoding="utf-8")
print("Saved:", REPORTS / "workspace_audit.html")
print("Attach this file to ChatGPT:", REPORTS / "milestone_summary.json")
print("Gate:", "READY FOR NEXT FEATURE INVESTIGATION" if audit["ready_for_next_feature_milestone"] else "STOP — RESOLVE PREFLIGHT GAPS")
print("No training, cloud provisioning, Kaggle submission or GitHub push occurred in this notebook.")
if REPORTS.is_relative_to(KIT):
    display(FileLink(str((REPORTS / "milestone_summary.json").relative_to(KIT))))
    display(FileLink(str((REPORTS / "workspace_audit.html").relative_to(KIT))))
assert audit["ready_for_next_feature_milestone"], "Preflight gaps remain; preserve this report and resolve them before fitting."